# Entity states v0.1: what research says about each entity type

**Question.** Before we model claims and arguments, what *states* does each ontology entity type (`src/incolor/ontology`) need, judging by what the research in `pdfs/` actually asserts about it?

**Method.**
1. Read all 80 corpus documents (`pdfs/*.txt`) and extracted 6 to 20 *core* propositions per document as `entity × attribute × value` rows, plus modality, horizon, driver and a verbatim quote. The rows are in `outputs/entity_states/propositions.jsonl` (LLM-assisted extraction, then hand curation in §1).
2. Grouped attributes into **state families** (§2) and checked how well subjects resolve to the registry (§3).
3. Reviewed each entity type and subtype (§4), **routed** every proposition to the entity that should *own* the state (§5), and drafted a state catalogue and a prototype `State`/`Claim` model (§6 and §7).
4. Listed ontology gaps (§8).

Working definition used throughout:

> **State** = *subject × attribute × value × period* (+ *reference* for relative attributes, + *participant* for positioning/flows).
> **Claim** = *State × modality × holder × source × as_of* (+ condition, + drivers).

`holder` is whoever holds the view: author, market, official, consensus or survey. That distinction drives most trade rationales ("the market prices X, we think Y").

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PDFS = ROOT / 'pdfs'
DATA = ROOT / 'outputs' / 'entity_states' / 'propositions.jsonl'

pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 200)

raw = pd.DataFrame([json.loads(line) for line in DATA.read_text(encoding='utf-8').splitlines()])


def front_matter(path: Path) -> dict:
    text = path.read_text(encoding='utf-8')
    match = re.match(r'---\n(.*?)\n---', text, re.S)
    return yaml.safe_load(match[1]) if match else {}


meta = pd.DataFrame([
    {'doc': p.stem, 'publisher': fm.get('Publisher'), 'doc_type': fm.get('Document type'), 'date': fm.get('Date')}
    for p in sorted(PDFS.glob('*.txt'))
    for fm in [front_matter(p)]
])
print(f'{len(raw)} propositions from {raw.doc.nunique()} of {len(meta)} documents')
meta.groupby(['publisher', 'doc_type']).size().rename('docs').to_frame().T

## 1. Curation

The extraction labels are mostly right. The fixes below cover what it got wrong or what the ontology cannot express yet:
- **Reclassified**: tenor-named bonds become instruments; strategies, factors and participant groups are moved out of `index`/`measure`; global regime talk moves to `geography: Global`; policy-rate paths stated about a country move to the central bank.
- **Gap labels** (flagged in §8): `tariff_trade` and `spending` policy kinds, `strategy`, `participant_group`, `asset_class`, `theme`.
- **Field repairs**: values placed in the wrong column.

In [ ]:
df = raw.copy()
df['curated'] = False


def fix(pattern: str, attr: str | None = None, **updates: str) -> None:
    mask = df.entity.str.fullmatch(pattern, case=False)
    if attr is not None:
        mask &= df.attribute.eq(attr)
    assert mask.any(), (pattern, attr)
    for column, value in updates.items():
        df.loc[mask, column] = value
    df.loc[mask, 'curated'] = True


# reclassify
fix(r'El Ni.o', etype='event', subtype='climate')
fix(r'PM Takaichi', subtype='public_official')
fix(r'Portfolio 365|DB Dynamic Multi-Factor|quality factor|value factor|carry trade|equity strategies', etype='other', subtype='strategy')
fix(r'Hedge Fund VIP.*|US equity long/short hedge.*|hedge funds|mutual funds|crowding in equity.*', etype='other', subtype='participant_group')
fix(r'EM positioning', etype='other', subtype='participant_group', attribute='positioning')
fix(r'bonds|equity|FX volatility', etype='other', subtype='asset_class')
fix(r'US tariffs|EU trade defense instruments', subtype='tariff_trade')
fix(r'UK public financial institution.*|UK National Wealth Fund|UK British Business Bank|UK defence spending|China FAI policy supports|RMB 800bn policy financing.*', subtype='spending')
fix(r'China special local govern.*', subtype='debt_management')
fix(r'UK Autumn Budget', etype='event', subtype='fiscal')
fix(r'European natural gas', etype='commodity', subtype='')
fix(r'US sentiment', etype='indicator', subtype='activity')
fix(r'DBSI', etype='index', subtype='')
fix(r'Fed pain threshold', etype='organization', subtype='central_bank', entity='Fed', attribute='reaction_function')
fix(r'stagflation risk|macro regime|global rate market cycle', etype='geography', subtype='market_grouping', entity='Global', attribute='regime')
fix(r'global monetary policy', etype='geography', subtype='market_grouping', entity='Global', attribute='policy_stance')
fix(r'monetary debasement|fiscal debasement|AI technology|anchoring themes.*', subtype='theme')
fix(r'global developed markets', subtype='market_grouping')
fix(r'China', attr='policy_path', etype='organization', subtype='central_bank', entity='PBOC')
fix(r'Brazil', attr='policy_stance', etype='organization', subtype='central_bank', entity='BCB')
fix(r'OAT-Bund spread|eurozone spreads over Bunds', etype='instrument', subtype='spread')
fix(r'UST 10y|10y gilts|Chinese 10y government bond|Japan 30y government bond', etype='instrument', subtype='outright')
fix(r'IMF', etype='currency', subtype='', entity='USD', attribute='flows')
fix(r'RBI', attr='policy_tool', etype='policy', subtype='liquidity_operation', entity='RBI FCNR/ECB measures', attribute='tool_setting')
fix(r'BoJ', attr='trend', attribute='policy_path')

# repair misplaced fields
fix(r'US core inflation ex-energy', attribute='level')
fix(r'US CPI', attr='observation', attribute='surprise_vs_consensus')
fix(r'inflation transmission', attribute='inflation_outlook', modality='observation')
fix(r'Treasury TGA', etype='indicator', subtype='financial', attribute='level')
fix(r'Australia', attr='attribute', attribute='fiscal_stance')
fix(r'Australia', attr='indicator', attribute='labour_market_slack')
fix(r'German government', attr='interpretation', attribute='fiscal_stance')
fix(r'German fiscal reform', attr='interpretation', attribute='tool_setting')
fix(r'EU', attr='fiscal_stance', attribute='external_balance')
misplaced = df.modality.eq('risk_balance')
df.loc[misplaced, ['attribute', 'modality', 'curated']] = ['risk_balance', 'forecast', True]
df.loc[df.subtype.eq('data_quality'), 'subtype'] = 'labour'

df = df.merge(meta, on='doc', how='left')
print(f"curated {df.curated.sum()} of {len(df)} rows")
df.etype.value_counts().to_frame().T

## 2. State families

The extraction used about 50 attribute names. They collapse into **ten families**. Seven describe a state of one entity. `CAUSAL` and `SPEECH` relate a state to something else, and `TRADE` describes a position rather than the instrument itself.

| Family | Question it answers | Value shape | Unary state? |
|---|---|---|---|
| `LEVEL` | how much, now or at a period | number + unit, or bucket | yes |
| `CHANGE` | which way, how fast | direction / Δ / run-rate | yes |
| `DECISION` | what an authority does at an event | discrete action (+ size) | yes, anchored to an event |
| `RELATIVE` | compared with what | Δ vs *reference* (consensus, target, CB forecast, fair value) | yes, needs a reference |
| `STANCE` | qualitative condition | ordinal or categorical label | yes |
| `UNCERTAINTY` | how sure, which way the risks lean | skew / probability / vol | yes |
| `FLOW_POSITION` | who holds or moves it | amount/label + **participant** | yes, needs a participant |
| `CAUSAL` | what moves it | edge: driver → state | **no**: argument layer |
| `SPEECH` | what an official said | nested claim | **no**: claim with holder = person |
| `TRADE` | the house's position | direction, levels, status, P&L | **no**: Position object |

In [ ]:
FAMILY = {
    'LEVEL': ['level', 'forecast_value', 'issuance', 'market_pricing', 'fiscal_rules_headroom', 'external_balance'],
    'CHANGE': ['change', 'trend', 'momentum', 'forecast_path', 'performance'],
    'DECISION': ['policy_decision', 'policy_path', 'event_outcome', 'tool_setting', 'operational_framework', 'personnel'],
    'RELATIVE': ['surprise_vs_consensus', 'surprise_vs_cb_forecast', 'valuation', 'carry_roll'],
    'STANCE': ['policy_stance', 'fiscal_stance', 'regime', 'growth_outlook', 'inflation_outlook', 'labour_market_slack',
               'credibility_independence', 'guidance_communication', 'credit_rating', 'regulatory_treatment', 'data_quality',
               'reaction_function'],
    'UNCERTAINTY': ['risk_balance', 'scenario_probability', 'volatility'],
    'FLOW_POSITION': ['positioning', 'flows', 'supply_demand', 'liquidity'],
    'CAUSAL': ['drivers_sensitivity', 'market_impact', 'correlation_or_beta'],
    'SPEECH': ['view_of_official', 'vote_split'],
    'TRADE': ['trade_recommendation', 'trade_levels', 'trade_status'],
}
ATTR_FAMILY = {a: fam for fam, attrs in FAMILY.items() for a in attrs}
df['family'] = df.attribute.map(ATTR_FAMILY)
unmapped = df[df.family.isna()]
assert unmapped.empty, unmapped[['entity', 'attribute']]

FAMILY_ORDER = list(FAMILY)
TYPE_ORDER = ['organization', 'person', 'geography', 'indicator', 'measure', 'policy', 'event',
              'currency', 'fx_pair', 'rate_benchmark', 'bond', 'index', 'commodity', 'instrument', 'other']


def heatmap(table: pd.DataFrame, title: str, ax=None) -> None:
    ax = ax or plt.subplots(figsize=(1.0 * table.shape[1] + 2, 0.42 * table.shape[0] + 1.2))[1]
    ax.imshow(table.values, cmap='Blues', aspect='auto')
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=40, ha='right')
    ax.set_yticks(range(table.shape[0]), table.index)
    for (i, j), v in pd.DataFrame(table.values).stack().items():
        if v:
            ax.text(j, i, int(v), ha='center', va='center', fontsize=8,
                    color='white' if v > table.values.max() * 0.6 else 'black')
    ax.set_title(title)


ct = pd.crosstab(df.etype, df.family).reindex(index=TYPE_ORDER, columns=FAMILY_ORDER, fill_value=0)
heatmap(ct, 'Propositions: entity type x state family')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
heatmap(pd.crosstab(df.family, df.modality).reindex(FAMILY_ORDER).fillna(0).astype(int), 'Family x modality', axes[0])
heatmap(pd.crosstab(df.doc_type, df.family).reindex(columns=FAMILY_ORDER, fill_value=0), 'Document type x family', axes[1])
plt.tight_layout()

**Reading the overview.**
- **Central banks dominate** the subjects, and almost all of it is `DECISION` (decision and path) or `STANCE`. The core proposition of macro research is *"what will authority X do, and why"*.
- **Indicators** are the numeric backbone: `LEVEL` (prints, forecasts), `CHANGE` (trend), `RELATIVE` (surprise vs consensus) and `UNCERTAINTY` (risk balance).
- **Markets** split in two. FX pairs and instruments are mostly `TRADE`. Bonds, commodities and rate benchmarks are `LEVEL`, i.e. pricing. Explicit `valuation` ("cheap", "rich") is rare. Instead, trade ideas compare the *author's* `DECISION` forecast with the *market's* `LEVEL` (`market_pricing`). That makes `holder` a first-class field on claims.
- **Trade ideas** are mostly `TRADE` rows. Their *rationale* rows (CB path, prints) look exactly like the rows in economics notes. A shared state vocabulary therefore links strategy to economics.
- `CAUSAL` rows exist for every market type. They are edges between states (oil → CLPCOP), not states.

## 3. Do the subjects resolve to the registry?

A rough check: resolve the full entity string, then each token (so `UST 2s10s` falls back to `UST`). This shows which state subjects have a registry anchor today.

In [ ]:
from incolor.ontology.registry import Registry

registry = Registry.load_default()


def anchor(name: str) -> str:
    for text in [name, *re.split(r'[\s/()-]+', name)]:
        if text and (hits := registry.resolve(text)):
            return ','.join(e.entity_id for e in hits)
    return ''


df['anchor'] = df.entity.map(anchor)
coverage = (df.assign(resolved=df.anchor.ne(''))
              .groupby('etype').resolved.agg(['mean', 'size'])
              .reindex(TYPE_ORDER).rename(columns={'mean': 'resolved_share', 'size': 'rows'}))
display(coverage.style.format({'resolved_share': '{:.0%}'}))
print('Unresolved subjects (sample):')
print(sorted(df.loc[df.anchor.eq(''), 'entity'].unique())[:60])

Organisations, currencies and FX pairs resolve well. The unresolved subjects show what a state needs as its subject besides a registry entity:
- **composite subjects**: instruments (`AUD 2y2y vs USD 2y2y`), dated events (`FOMC Dec-26`), meeting-dated pricing (`Nov RBA OIS`);
- **un-registered series**: `US real spending growth`, `UK gross financing requirement`;
- **groups and narratives**: hedge funds, CTAs, "fiscal debasement".

So a state's `subject` must accept **entity id | instrument key | event occurrence (series + date) | theme id**. That matches the `subject: str  # entity id or instrument key` comment in `ontology/notes.txt`.

## 4. Entity types one by one

`show()` prints, for a type or subtype, the attribute × modality counts and example propositions. The reflection after each block proposes the states that type should carry.

In [ ]:
COLUMNS = ['entity', 'attribute', 'modality', 'value', 'horizon', 'driver', 'quote']


def show(etype: str, subtype: str | None = None, n: int = 10, by: str = 'subtype') -> None:
    rows = df[df.etype.eq(etype)]
    if subtype is not None:
        rows = rows[rows.subtype.eq(subtype)]
    print(f'{etype}{"/" + subtype if subtype else ""}: {len(rows)} propositions, '
          f'{rows.entity.nunique()} subjects, {rows.doc.nunique()} docs')
    if subtype is None and rows[by].nunique() > 1:
        display(rows[by].value_counts().to_frame().T)
    display(pd.crosstab(rows.attribute, rows.modality, margins=True).sort_values('All', ascending=False))
    sample = rows.sort_values(['family', 'attribute']).groupby('attribute', group_keys=False).head(2)
    display(sample[COLUMNS].head(n).reset_index(drop=True))

### 4.1 Organisation: central banks

In [ ]:
show('organization', 'central_bank', n=16)

**Reflection: central banks.** Two kinds of proposition are mixed under the CB name:

1. **What happens to the policy rate**: `policy_decision` and `policy_path` ("hike 25bp in Dec, cut Jun-27", "terminal 4.25%"). These are **states of the policy-rate indicator** (`meas:policy_rate × geo`), anchored to an **event occurrence** (FOMC Dec-26). Storing them there lets the author's path, the CB's own projection (dots) and market pricing (OIS) sit on **one subject** with different *holders*, which is exactly what the AUD/RBA trade ideas exploit. This matches the `ind:fed_funds_target` example in `notes.txt`.
2. **What the institution is like**: these stay on the organisation:

| CB state | Value domain | Example |
|---|---|---|
| `stance` | ordinal: very dovish … neutral … very hawkish (+ "behind the curve") | "RBA hawkish rhetoric", "BoC hawkish hold" |
| `policy_setting` | restrictive / neutral / accommodative (vs the neutral rate) | "policy already restrictive before Sept hike" |
| `reaction_function` | weights or thresholds on indicators (inflation, labour, FCI, FX) | "Fed pain threshold very low" |
| `guidance` | explicit / data-dependent / none, + bias | "no hike, no guidance", "every meeting live" |
| `credibility` | intact / questioned (+ independence) | "ECB reaction function unstable" |
| `balance_of_risks` (holder = CB) | upside / balanced / downside | "BoJ: upside price risks now equal downside" |
| `leadership` | person-role-tenure changes | "ECB President departure imminent" |

`stance` is the most reused summary state and should be **ordinal** so that change ("shifting to a moderate path", "another step up") becomes a `CHANGE` of an ordinal state.

In [ ]:
rows = df[df.etype.eq('organization') & df.subtype.ne('central_bank')]
display(rows[['subtype', *COLUMNS]].reset_index(drop=True))

**Reflection: other organisations.** Very few propositions have a non-CB organisation as *subject*. Organisations appear mostly as **holders** (Moody's view, IMF data, survey respondents) or as **participants** (pension funds, reserve managers). Proposed states:
- `government`: `fiscal_stance` (tight / neutral / loose; "do no harm"), `policy_continuity`. Numeric fiscal states belong on fiscal indicators.
- Credit rating is a state of the **issuer's bond curve** (`bond:bund`) or the issuer org, with the rating agency as *holder*.
- `regulatory_treatment` (risk weight 20%) is a state of a **bond market or segment × jurisdiction**, not of the investor.

### 4.2 People

In [ ]:
show('person', n=14)

**Reflection: people.** About half are **speech**: "Jefferson: FOMC may need more time". Their content is a claim about *another* subject (the policy rate), with `holder = person`. A person's own states are few:

| Person state | Value domain | Example |
|---|---|---|
| `stance` | hawk-dove ordinal (as for CBs) | "Logan hawkish", "MPC swing voters moved hawkish" |
| `vote` | dissent / with majority, per event | "dissents from Dallas, Cleveland likely" |
| `role` | office + start/end | "Warsh as Chair", "President departure" |

Political figures (Takaichi, Lula, Merz) carry `political_capital` or `stance` toward policy, plus election outcomes, which are better modelled as an **event outcome**.

### 4.3 Indicators (by measure kind)

In [ ]:
show('indicator', 'prices', n=16)

In [ ]:
for kind in ['labour', 'activity']:
    show('indicator', kind, n=10)
display(df[df.etype.eq('indicator') & ~df.subtype.isin(['prices', 'labour', 'activity'])][['subtype', *COLUMNS]])

**Reflection: indicators.** This is the most regular type. Every proposition is a value of one series, either observed or forecast, read through a *transform* and often against a *reference*:

| Indicator state | Shape | Qualifiers that must be explicit |
|---|---|---|
| `value` | number + unit | **reference period** (Sep-26, Q4, 2027), transform (y/y, m/m, 3m saar), basket, SA/NSA |
| `surprise` | value − reference | reference ∈ {consensus, prior, CB forecast, own forecast} |
| `target_gap` | above / at / below target (+ persistence) | target (CB-specific) |
| `trend` / `momentum` | direction + run-rate | window ("since July", "3m annualised") |
| `risk_balance` | upside / balanced / downside | horizon |
| `regime label` (measure-kind specific) | categorical | e.g. prices: *sticky, broadening, second-round*; labour: *tight, balanced, loosening*; activity: *above/below trend* |
| `data_quality` | categorical | *seasonal distortion, revision, shutdown gap* |

Design points:
- **Reference period ≠ as_of.** "UK CPI above 4% in Q1-27" is a *forecast* of the Q1-27 value made on the publication date. A forecast and the later print are the **same state** (subject × attribute × period). That makes forecast-vs-outcome evaluation trivial.
- `surprise` needs a **reference** value, the same pattern as `valuation` for markets (§4.7).
- Kind-specific labels: a small enum per `MeasureKind` beats free text and keeps the generic schema.
- **Monetary** (policy rate) indicators absorb CB decisions and paths (§4.1). **Fiscal** indicators absorb the numeric fiscal states from geography/government (deficit, financing requirement, headroom vs rule).

### 4.4 Measures and themes

In [ ]:
show('measure', n=20)

**Reflection: measures.** No proposition is truly about a country-free *concept*. Each is either:
- an implicit **global or US series** ("term premium higher", "financial conditions tightening ≈ 25bp hike"). Resolve to an `Indicator` with `geo:global` / `geo:us` and give it ordinary indicator states. **Measures should not carry states.**
- a **narrative / theme** ("fiscal debasement: rising concern", "AI as near-term inflation pressure"). These fit the existing `Theme` (`ontology/theme.py`). Theme states are `salience` (rising / fading / dominant) and `direction of effect` on other states, which is causal.

### 4.5 Geography

In [ ]:
show('geography', n=24)

**Reflection: geography.** Writers use the country as shorthand. Most rows have a better home:
- `growth_outlook` → GDP/activity indicator; `inflation_outlook` → CPI; `labour_market_slack` → unemployment / labour indicator; `fiscal_stance` / headroom → fiscal indicators or the government.

What is left is truly **macro-regime**-level, and is worth a small, explicit state:

| Geography state | Value domain | Example |
|---|---|---|
| `macro_regime` | growth {above, at, below trend} × inflation {above, at, below target} → goldilocks / reflation / stagflation / disinflationary slowdown | "stagflation risk rising from reflation", "cycle like no other" |
| `political_risk` | low / elevated / acute (+ event link) | France, Brazil run-off, US midterms |
| `structural_theme` | link to `Theme` + direction | "AI investment surge", "funding model breakdown risk" |

`macro_regime` can be *derived* from indicator states, but authors also assert it directly, so it must be storable as a claim.

### 4.6 Policy tools and events

In [ ]:
show('policy', n=20)

**Reflection: policy tools.** `policy.py` already says a tool's *setting or status is a state*. The data confirms three states:

| Policy-tool state | Value domain | Example |
|---|---|---|
| `status` | proposed → announced → active → expanded / reduced / paused → ended; plus deferred / rejected | "CGT change deferred", "liquidity backstop introduced", "TPI not imminent" |
| `parameters` | size, rate, pace, eligibility (number + unit) | "RMB500bn special LGB", "RRP 1.25%", "GBP 20.4bn PuFins" |
| `deployment_probability` | probability or likely / unlikely, often conditional | "ECB may deploy TPI for France if needed" |

Market effects ("MoF intervention short-lived") are `CAUSAL` edges from a tool state to a market state. Gaps: no `PolicyKind` fits **tariffs / trade defence** or **public spending / investment / policy-bank financing** (§8).

In [ ]:
show('event', n=16)

**Reflection: events.** The registry holds *series* (FOMC). Propositions are about **occurrences** (FOMC Oct-26). The state subject must therefore be `event series + date`. Event states:

| Event state | Value domain | Example |
|---|---|---|
| `timing` | date, or "live / not live" | "every BoJ meeting is live" |
| `outcome_distribution` | {outcome: probability}, by holder (author, market, survey) | "hold 78% (market)", "hike 25bp, 86%" |
| `outcome` | realised outcome | "Lula re-elected", "hike 25bp delivered" |
| `surprise` | realised vs distribution | "hawkish surprise" |
| `tone` | for meetings/publications: hawkish / neutral / dovish | "neutral tone expected" |

For **policy meetings**, the decision itself is a state of the policy-rate indicator *at* that event (§4.1). The event keeps only the probability distribution, the tone and the surprise. For **non-policy events** (elections, budgets, conflicts) the outcome lives on the event, and its effects are causal edges (Hormuz → Brent; run-off → BRL, DI).

### 4.7 Markets

In [ ]:
for etype in ['currency', 'fx_pair']:
    show(etype, n=14)

**Reflection: currency vs FX pair.**
- **Currency** propositions are about *broad* value and who trades it: `direction` (vs a basket), `flows` (reserve managers, active selling ~USD15bn/month), `positioning` (moderately short USD), `regime` (market-determined vs managed, intervention zone, e.g. "USDCNY appreciation capped ~7"), `volatility`.
- **FX pair** propositions are about *price*: `spot level / target`, `direction`, `volatility`, `positioning / gamma` ("EURUSD anchored by gamma"), and a lot of `drivers` (carry spread, oil, rate differentials). Those are causal edges. The trade rows go to the Position object.
- Currency states need a **participant** (who holds or flows), so `FLOW_POSITION` states get `participant` as part of their key.

In [ ]:
for etype in ['rate_benchmark', 'bond']:
    show(etype, n=14)

**Reflection: rates.**
- **Rate benchmarks** carry the **market-implied policy path**: bp priced per meeting ("22.5bp for Nov RBA"), terminal rate priced ("ECB terminal 3.25% feels excessive"), forwards vs CB guidance ("BoJ 1y1y +28bp above"). This is the *market holder's* version of the indicator path in §4.1. The trade rationale is the **gap** between them, a `RELATIVE` state whose reference is the author's (or CB's) path.
- **Gap**: `Tenor` cannot express **meeting-dated** (Nov RBA, Dec/Feb RBA spread) or **contract-dated** (SFRZ6Z7, Dec26/27 SONIA) points. Instruments need an event- or IMM-anchored period.
- **Bonds**: once tenor-specific rows are instruments, the curve-level states are `valuation` (overwide, cheap vs fundamentals), `term_premium`, `supply_demand` (issuance, remit, index inclusion flows into IGBs), `credit_rating` (holder = agency) and `regulatory_treatment`. Yield and spread *levels* are states of instruments (`UST 10y`, `10y OAT-Bund`).

In [ ]:
for etype in ['commodity', 'index']:
    show(etype, n=14)
show('other', n=20)

**Reflection: commodities, indices and the rest.**
- **Commodities**: `level` and `target` (often **conditional** on a scenario: "USD120 Brent if Bab el-Mandeb blockade", "gold $4,800 if Hormuz reopens"), `supply_balance`, `positioning` ("refined energy very long, concentrated"), `volatility`. Conditional targets need `Claim.condition` pointing to an event state.
- **Indices**: `level/performance`, `momentum`, `positioning`, `earnings`. Valuation is surprisingly absent in this FX/rates-heavy corpus.
- **Other**: three concepts the ontology lacks.
  - **Participant groups** (hedge funds, CTAs, mutual funds, reserve managers, EM investors) are *who* in positioning and flow states.
  - **Strategies / factors** (Portfolio 365, value, quality, carry) have `performance`, `drivers`, `crowding`.
  - **Asset classes** as subjects ("bonds: CTAs very short", "equity moderately long").

### 4.8 Instruments and trades

In [ ]:
show('instrument', n=20)
trades = df[df.family.eq('TRADE')]
print('TRADE rows by entity type:')
trades.etype.value_counts().to_frame().T

**Reflection: instruments vs positions.** Two different things are in this table:

1. **Market states of an instrument**:
   - `level` (yield, spread, slope);
   - `direction`, in structure-specific words: steeper / flatter, wider / tighter, richer / cheaper;
   - `valuation` vs a reference (fair value, history, model);
   - `carry_roll`;
   - `volatility`.

   The vocabulary depends on `Structure`: `CURVE` → steepen/flatten; `SPREAD` / `CROSSFLY` → widen/tighten; `OUTRIGHT` → up/down. So the direction enum should be chosen from the structure.
2. **The house's position**: direction, size (DV01 / notional), entry, target, stop, status (open → resized → closed / stopped / target hit), P&L, *rationale*. This is not a state of the instrument. It is a **Position / Recommendation** object owned by the publisher, with its own lifecycle across documents. BNP trade ideas are almost entirely updates to this object. Its rationale links to the claims above (e.g. "receive Nov RBA" ← {author: RBA hold; market: 22.5bp priced}).

## 5. Routing: where each state should live

The rules below apply §4 to every proposition. A row's *home* is the subject that should own the state after routing.

In [ ]:
GEO_TO_INDICATOR = {'growth_outlook': 'activity', 'inflation_outlook': 'prices', 'labour_market_slack': 'labour',
                    'fiscal_stance': 'fiscal', 'fiscal_rules_headroom': 'fiscal', 'external_balance': 'external'}


def home(r: pd.Series) -> str:
    if r.family == 'TRADE':
        return 'position'
    if r.family == 'CAUSAL':
        return 'causal edge'
    if r.family == 'SPEECH' and r.etype == 'person':
        return 'claim(holder=person)'
    if r.attribute in ('policy_decision', 'policy_path', 'event_outcome') and r.subtype in ('central_bank', 'policy_meeting'):
        return 'indicator:monetary'
    if r.etype == 'geography' and r.attribute in GEO_TO_INDICATOR:
        return 'indicator:' + GEO_TO_INDICATOR[r.attribute]
    if r.etype == 'measure':
        return 'theme' if r.subtype == 'theme' else 'indicator:global'
    if r.etype == 'indicator':
        return 'indicator:' + r.subtype
    if r.etype in ('organization', 'other', 'policy', 'event', 'person'):
        return f'{r.etype}:{r.subtype}'
    return r.etype


df['home'] = df.apply(home, axis=1)
routing = pd.crosstab(df.etype, df.home).reindex(index=TYPE_ORDER, fill_value=0)
heatmap(routing.loc[:, routing.sum().sort_values(ascending=False).index], 'Mentioned type (rows) -> state home (columns)')
plt.tight_layout()

In [ ]:
state_rows = df[~df.home.isin(['position', 'causal edge', 'claim(holder=person)'])]
print(f'{len(state_rows)} unary states, {df.home.eq("position").sum()} position updates, '
      f'{df.home.eq("causal edge").sum()} causal edges, {df.home.eq("claim(holder=person)").sum()} attributed speech')
by_home = pd.crosstab(state_rows.home, state_rows.family)
by_home = by_home.loc[by_home.sum(axis=1).sort_values(ascending=False).index]
heatmap(by_home.reindex(columns=[f for f in FAMILY_ORDER if f in by_home.columns]), 'State home x family (unary states only)')
plt.tight_layout()

After routing, the **policy-rate indicator** becomes the single biggest state subject. It merges the CB-decision, meeting-outcome and path propositions that were spread over organisation, event and person mentions. CB organisations keep only `STANCE`-type states. Geography shrinks to regime and political risk.

## 6. Proposed state catalogue

One row per (home, state). This is the main deliverable of the notebook. The `qualifiers` column lists what must be explicit for two claims to be about *the same* state, so that they can agree, conflict or be checked against outcomes.

In [ ]:
CATALOGUE = [
    # home, state, family, value domain, qualifiers, example
    ('organization:central_bank', 'stance', 'STANCE', 'ordinal -2..+2 (dovish..hawkish) + behind_curve flag', 'as_of', 'RBA hawkish rhetoric'),
    ('organization:central_bank', 'policy_setting', 'STANCE', 'restrictive | neutral | accommodative', 'as_of, reference=neutral rate', 'policy already restrictive'),
    ('organization:central_bank', 'reaction_function', 'STANCE', '{indicator: weight/threshold}', 'as_of', 'Fed pain threshold very low'),
    ('organization:central_bank', 'guidance', 'STANCE', 'explicit | data_dependent | none (+ bias)', 'event', 'no hike, no guidance'),
    ('organization:central_bank', 'credibility', 'STANCE', 'intact | questioned (+ independence)', 'as_of', 'ECB reaction function unstable'),
    ('organization:central_bank', 'leadership', 'DECISION', 'person x role x start/end', 'date', 'ECB President departure'),
    ('organization:government', 'fiscal_stance', 'STANCE', 'tight | neutral | loose', 'fiscal year / event', 'Budget: do no harm'),
    ('person', 'stance', 'STANCE', 'ordinal -2..+2', 'as_of', 'Logan hawkish'),
    ('person', 'vote', 'DECISION', 'majority | dissent (direction)', 'event', 'dissents likely'),
    ('person', 'role', 'DECISION', 'office x start/end', 'date', 'Warsh as Chair'),
    ('indicator:monetary', 'decision', 'DECISION', 'hike | cut | hold x size_bp', 'event occurrence', 'hike 25bp Dec FOMC'),
    ('indicator:monetary', 'path', 'DECISION', '[(event|period, change_bp)]', 'horizon', 'hike Dec, cut Jun-27'),
    ('indicator:monetary', 'terminal', 'LEVEL', 'rate % + timing', 'horizon', 'terminal 4.25%'),
    ('indicator:monetary', 'priced (holder=market)', 'LEVEL', 'bp per meeting / terminal %', 'event occurrence, as_of', '22.5bp priced Nov RBA'),
    ('indicator:*', 'value', 'LEVEL', 'number + unit', 'period, transform, basket, sa', 'US CPI 3.7% y/y Sep'),
    ('indicator:*', 'surprise', 'RELATIVE', 'value - reference', 'period, reference in {consensus, prior, cb_forecast}', '+0.6% m/m vs cons'),
    ('indicator:prices', 'target_gap', 'RELATIVE', 'above | at | below (+ persistence)', 'period, reference=target', 'above target for extended period'),
    ('indicator:*', 'trend', 'CHANGE', 'up | flat | down + run-rate', 'window', 'trimmed mean ~3% annualised'),
    ('indicator:*', 'risk_balance', 'UNCERTAINTY', 'upside | balanced | downside', 'horizon', 'upside, second-round effects'),
    ('indicator:*', 'condition', 'STANCE', 'kind-specific enum (sticky/broadening; tight/loosening; above/below trend)', 'as_of', 'labour market stable'),
    ('indicator:*', 'data_quality', 'STANCE', 'seasonal | revision | gap', 'period', 'seasonals more punitive'),
    ('indicator:fiscal', 'headroom', 'RELATIVE', 'amount vs fiscal rule', 'fiscal event', 'headroom ~GBP16bn'),
    ('geography', 'macro_regime', 'STANCE', 'growth x inflation quadrant', 'horizon', 'stagflation risk rising'),
    ('geography', 'political_risk', 'UNCERTAINTY', 'low | elevated | acute (+ event)', 'horizon', 'French election risk'),
    ('theme', 'salience', 'CHANGE', 'rising | dominant | fading', 'as_of', 'fiscal debasement rising concern'),
    ('policy', 'status', 'DECISION', 'proposed > announced > active > expanded/reduced/paused > ended; deferred', 'date / event', 'CGT change deferred'),
    ('policy', 'parameters', 'LEVEL', 'size | rate | pace | eligibility', 'date', 'RMB500bn special LGBs'),
    ('policy', 'deployment_probability', 'UNCERTAINTY', 'probability | likely/unlikely', 'condition', 'TPI if needed for France'),
    ('event', 'timing', 'LEVEL', 'date | live/not live', '-', 'every BoJ meeting live'),
    ('event', 'outcome_distribution', 'UNCERTAINTY', '{outcome: p}', 'holder, as_of', 'hold 78% priced'),
    ('event', 'outcome', 'DECISION', 'realised outcome', 'date', 'Lula re-elected'),
    ('event', 'tone', 'STANCE', 'hawkish | neutral | dovish', '-', 'neutral tone expected'),
    ('currency', 'direction', 'CHANGE', 'appreciate | depreciate (vs basket)', 'horizon', 'GBP weaker into year-end'),
    ('currency', 'regime', 'STANCE', 'free | managed | intervention zone', 'as_of', 'USDCNY capped ~7'),
    ('fx_pair', 'spot / target', 'LEVEL', 'price', 'horizon', 'USDJPY 155-160'),
    ('fx_pair', 'direction', 'CHANGE', 'up | down', 'horizon', 'USDJPY higher'),
    ('market:*', 'positioning', 'FLOW_POSITION', 'long | short | neutral (+ extremity)', 'participant, as_of', 'CTAs very short bonds'),
    ('market:*', 'flows', 'FLOW_POSITION', 'amount + direction', 'participant, period', 'reserve managers bought USD'),
    ('market:*', 'volatility', 'UNCERTAINTY', 'level | higher/lower', 'tenor, horizon', 'USDINR vol lower'),
    ('rate_benchmark', 'implied path', 'LEVEL', 'forward rates by meeting/tenor', 'meeting-dated tenor, as_of', 'BoJ 1y1y 2.03%'),
    ('bond', 'valuation', 'RELATIVE', 'rich | fair | cheap', 'reference (fair value / history)', 'OAT spreads overwide'),
    ('bond', 'supply_demand', 'FLOW_POSITION', 'issuance / index inclusion amount', 'period', 'IGB inclusion USD15-25bn'),
    ('bond', 'credit_rating', 'STANCE', 'rating + outlook', 'holder=agency', 'Germany AAA stable'),
    ('bond', 'regulatory_treatment', 'STANCE', 'risk weight / eligibility', 'jurisdiction', '20% risk weight CRR'),
    ('commodity', 'level / target', 'LEVEL', 'price', 'horizon, condition', 'Brent USD120 if blockade'),
    ('index', 'performance', 'CHANGE', 'return %', 'window', 'S&P flat -8bp w/w'),
    ('instrument', 'level', 'LEVEL', 'yield | spread | slope', 'as_of', '10y OAT-Bund 150bp'),
    ('instrument', 'direction', 'CHANGE', 'structure-specific: steeper/flatter, wider/tighter, up/down', 'horizon', 'OAT-Bund widen to 100bp'),
    ('instrument', 'valuation', 'RELATIVE', 'rich | fair | cheap', 'reference', '5y5y Bund expensive'),
    ('instrument', 'carry_roll', 'LEVEL', 'bp per period', 'period', '+13bp annualised'),
    ('position (new)', 'lifecycle', 'TRADE', 'direction, size, entry, target, stop, status, P&L, rationale -> claims', 'owner, instrument', 'receive Nov RBA, doubled to USD30k/01'),
    ('other:strategy (new)', 'performance / crowding', 'CHANGE', 'return %, crowding level', 'window', 'Portfolio 365 +13% YTD'),
]
catalogue = pd.DataFrame(CATALOGUE, columns=['home', 'state', 'family', 'value_domain', 'qualifiers', 'example'])
catalogue.style.hide(axis='index').set_properties(**{'text-align': 'left'})

## 7. Prototype: `State` and `Claim`

A minimal model that extends the sketch in `ontology/notes.txt` with the fields the corpus needs: `period` separate from `as_of`, `reference`, `participant`, `holder` and `condition`. As a fit test, every proposition is mapped onto it. A row fails if its family needs a field that the extraction did not capture.

In [ ]:
from dataclasses import dataclass, field
from datetime import date
from enum import StrEnum


class Holder(StrEnum):
    AUTHOR = 'author'
    MARKET = 'market'
    OFFICIAL = 'official'
    CONSENSUS = 'consensus'
    THIRD_PARTY = 'third_party'


class Modality(StrEnum):
    OBSERVATION = 'observation'
    INTERPRETATION = 'interpretation'
    FORECAST = 'forecast'
    CONDITIONAL = 'conditional'
    RECOMMENDATION = 'recommendation'


@dataclass(frozen=True, kw_only=True)
class State:
    subject: str                    # entity id | instrument key | event occurrence | theme id
    attribute: str                  # from the catalogue, e.g. 'stance', 'value', 'decision'
    value: str                      # numeric or enum, kept as text in this prototype
    period: str | None = None       # reference period the state is about (Sep-26, Dec FOMC), not when it was said
    reference: str | None = None    # RELATIVE only: consensus, target, cb_forecast, fair_value, market
    participant: str | None = None  # FLOW_POSITION only: CTAs, reserve managers


@dataclass(frozen=True, kw_only=True)
class Claim:
    state: State
    modality: Modality
    holder: Holder
    source: str
    as_of: date | None = None
    condition: str | None = None
    drivers: tuple[str, ...] = field(default_factory=tuple)


REFERENCE_FROM_ATTR = {'surprise_vs_consensus': 'consensus', 'surprise_vs_cb_forecast': 'cb_forecast'}


def holder_of(r: pd.Series) -> Holder:
    if r.attribute == 'market_pricing':
        return Holder.MARKET
    if r.modality == 'attributed':
        return Holder.OFFICIAL if r.etype in ('person', 'organization') else Holder.THIRD_PARTY
    return Holder.AUTHOR


def to_claim(r: pd.Series) -> Claim:
    state = State(subject=r.anchor or r.entity, attribute=r.attribute, value=r.value,
                  period=r.horizon or None, reference=REFERENCE_FROM_ATTR.get(r.attribute),
                  participant=r.entity if r.subtype == 'participant_group' else None)
    modality = Modality.INTERPRETATION if r.modality == 'attributed' else Modality(r.modality)
    return Claim(state=state, modality=modality, holder=holder_of(r), source=r.doc,
                 as_of=r.date, condition=r.driver if r.modality == 'conditional' else None,
                 drivers=(r.driver,) if r.driver else ())


def gaps(r: pd.Series, c: Claim) -> list[str]:
    missing = []
    if r.family == 'RELATIVE' and c.state.reference is None:
        missing.append('reference')
    if r.family == 'FLOW_POSITION' and c.state.participant is None:
        missing.append('participant')
    if r.family in ('DECISION', 'LEVEL') and c.modality is Modality.FORECAST and c.state.period is None:
        missing.append('period')
    return missing


df['claim'] = df.apply(to_claim, axis=1)
df['missing'] = [gaps(r, c) for (_, r), c in zip(df.iterrows(), df.claim)]
units = df[~df.home.isin(['position', 'causal edge'])]
fit = units.explode('missing').missing.fillna('-').value_counts()
print(f'{len(units)} state claims; {units.missing.str.len().eq(0).mean():.0%} carry every field their family needs')
display(fit.to_frame('rows').T)
display(pd.crosstab(units.home, units.claim.map(lambda c: c.holder.value)).sort_values('author', ascending=False).head(12))
example = df.loc[df.entity.eq('Nov RBA OIS') & df.attribute.eq('market_pricing'), 'claim'].iloc[0]
example

**Reading the fit test.**
- **`period` is the most common missing field.** Many forecasts carry no explicit reference period ("Fed to hike", "oil elevated"). Extraction must resolve it from context: the next meeting, the document date + default horizon.
- **`reference`** is missing for `valuation` and `carry_roll`: "cheap" against *what*? The text rarely says, so the model should allow `reference = unstated` rather than force a value.
- **`participant`** is missing on positioning rows where the holder group is implicit ("positioning moderately short USD"). Default it to `market (aggregate)`.
- `holder ≠ author` covers only a minority of rows, but those rows (market pricing, officials, consensus, surveys) are what trade rationales are built on. Make the field required.

## 8. Ontology gaps and decisions to take

| # | Finding | Proposal |
|---|---|---|
| 1 | CB decisions/paths are mentioned under CB, meeting, person and country | Canonical home = **policy-rate indicator × event occurrence**. CB keeps stance, reaction function, guidance and credibility |
| 2 | `Tenor` cannot express meeting-dated (Nov RBA, Dec/Feb RBA) or contract-dated (SFRZ6Z7, Dec26/27 SONIA) points | Add an event- or IMM-anchored period to instruments (e.g. `MeetingTenor(event_id, date)`) |
| 3 | Event registry holds series; states need occurrences | `subject = event series + date` key, e.g. `EVT[event:fomc@2026-12]` |
| 4 | `PolicyKind` has no tariff/trade or spending/investment kinds | Add `TRADE_POLICY` (tariffs, trade defence) and `SPENDING` (public investment, policy-bank financing, PuFins) |
| 5 | Measures attract states with no geography | No states on `Measure`. Use `geo:global` / regional indicators (term premium, FCI, global inflation) |
| 6 | Narratives (debasement, AI, K-shape) have no subject | Use `Theme` as a state subject with a `salience` state |
| 7 | Positioning and flows need *who* | `participant`: new participant-group entity (CTAs, hedge funds, reserve managers, real money) or `OrgKind.MARKET_PARTICIPANT` groups |
| 8 | Strategies, factors and house portfolios are subjects in quant notes | New category `strategy` (out of scope for FX/rates v1?) |
| 9 | Trade updates are the majority of trade-idea propositions | New **Position** object with a lifecycle across documents; rationale links to claims. Not instrument state |
| 10 | Relative states need a reference; forecasts need a reference period | `State.reference`, `State.period` (≠ `Claim.as_of`); allow `unstated` |
| 11 | Views differ by holder (author vs market vs CB vs consensus) | `Claim.holder` required. The author-vs-market gap is the trade thesis |
| 12 | `CAUSAL` propositions (drivers, market impact) | Edges between states in the argument layer, not states |
| 13 | Stance-like states (hawkish, restrictive, tight) need comparability over time | Ordinal enums per state so that "another step up" is a computable change |

**Suggested next step.** Freeze the catalogue in §6 as a YAML vocabulary (`config/taxonomies/states.yaml`: home type → attribute → value domain → required qualifiers). Then rerun the extraction against that closed vocabulary and measure agreement with this open-vocabulary pass.